In [1]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd

In [2]:
df_fd = pd.read_csv('data/mlb/FanDuel-MLB-2022 ET-05 ET-06 ET-75739-players-list.csv')

In [3]:
#df.drop(df[(df['Unit_Price'] >400) & (df['Unit_Price'] < 600)].index, inplace=True)
pitchers_list = ['Yu Darvish', 'Chad Kuhl']
def filter_pitchers(df, pitchers_list = pitchers_list):
    df_fd_filtered = df.query("Position != 'P' | Nickname == @pitchers_list")
    return df_fd_filtered

df_fd_filtered = filter_pitchers(df_fd)
df_fd_filtered.head()

,Id,Position,First Name,Nickname,Last Name,FPPG,Played,Salary,Game,Team,Opponent,Injury Indicator,Injury Details,Tier,Probable Pitcher,Batting Order,Roster Position
12,75739-16931,P,Yu,Yu Darvish,Darvish,28.600000,5.0,10200,MIA@SD,SD,MIA,NaN,NaN,NaN,Yes,NaN,P
38,75739-68484,P,Chad,Chad Kuhl,Kuhl,34.250000,4.0,9000,COL@ARI,COL,ARI,NaN,NaN,NaN,Yes,NaN,P
553,75739-60643,OF,Aaron,Aaron Judge,Judge,14.162500,24.0,4500,TEX@NYY,NYY,TEX,O,Postponed,NaN,NaN,NaN,OF/UTIL
554,75739-38976,3B,Jose,Jose Ramirez,Ramirez,14.973076,26.0,4400,TOR@CLE,CLE,TOR,O,Postponed,NaN,NaN,NaN,3B/UTIL
555,75739-12933,OF,Mike,Mike Trout,Trout,13.317391,23.0,4300,WSH@LAA,LAA,WSH,NaN,NaN,NaN,NaN,NaN,OF/UTIL


In [4]:
df_fd_filtered.Team.unique()

array(['SD', 'COL', 'NYY', 'CLE', 'LAA', 'HOU', 'MIN', 'TB', 'TOR', 'PHI',
       'STL', 'WSH', 'ATL', 'BOS', 'SEA', 'NYM', 'TEX', 'MIA', 'MIL',
       'CWS', 'SF', 'BAL', 'DET', 'OAK', 'ARI', 'KC'], dtype=object)

In [5]:
team_list = ['TEX', 'NYY,', 'NYM', 'BAL', 'KC', 'BAL', 'TOR', 'CLE']
def exclude_teams_weather(df, team_list = team_list):
    df_fd_weather = df.query("Team != @team_list")
    return df_fd_weather
    
df_fd_weather = exclude_teams_weather(df_fd_filtered)
df_fd_weather.Team.unique()

array(['SD', 'COL', 'NYY', 'LAA', 'HOU', 'MIN', 'TB', 'PHI', 'STL', 'WSH',
       'ATL', 'BOS', 'SEA', 'MIA', 'MIL', 'CWS', 'SF', 'DET', 'OAK',
       'ARI'], dtype=object)

In [6]:
def exclude_bad_players(df, team_list = team_list):
    df_fd_final = df.query("Salary >= 2500")
    return df_fd_final
    
df_fd_final = exclude_bad_players(df_fd_weather)
df_fd_final.Salary.unique()

array([10200,  9000,  4500,  4300,  4200,  4100,  4000,  3900,  3800,
        3700,  3600,  3500,  3400,  3300,  3200,  3100,  3000,  2900,
        2800,  2700,  2600,  2500])

In [7]:
df_fd_final.to_csv('data/mlb/FD_proj.csv')

In [8]:
from pydfs_lineup_optimizer import Site, Sport, get_optimizer, PlayerFilter, AfterEachExposureStrategy, TeamStack, RandomFantasyPointsStrategy
optimizer = get_optimizer(Site.FANDUEL, Sport.BASEBALL)
optimizer.load_players_from_csv("data/mlb/FD_proj.csv")
#New 
optimizer.add_stack(TeamStack(3, max_exposure=0.5))
optimizer.restrict_positions_for_opposing_team(['P'], ['C', 'SS', 'OF', '1B', '2B', '3B'])

In [9]:
for lineup in optimizer.optimize(10, max_exposure=.75, exposure_strategy=AfterEachExposureStrategy):
    print(lineup)
optimizer.export('data/mlb/fd_result.csv')

 1. P       Chad Kuhl                     P     COL            COL@ARI  34.25          9000.0$   
 2. C/1B    Rowdy Tellez                  1B    MIL            MIL@ATL  11.96          3000.0$   
 3. 2B      Ha-Seong Kim                  2B/SS SD             MIA@SD   10.465         2600.0$   
 4. 3B      Luis Urias                    2B/3B/SSMIL            MIL@ATL  17.6           2600.0$   
 5. SS      Jazz Chisholm Jr.             2B/SS MIA            MIA@SD   14.576         3600.0$   
 6. OF      Christian Yelich              OF    MIL            MIL@ATL  11.744         3300.0$   
 7. OF      Ramon Laureano                OF    OAK            OAK@MIN  10.308         2600.0$   
 8. OF      Taylor Ward                   3B/OF LAA            WSH@LAA  16.058         4000.0$   
 9. UTIL    Manny Machado                 3B    SD             MIA@SD   16.038         4200.0$   

Fantasy Points 143.00
Salary 34900.00

 1. P       Yu Darvish                    P     SD             MIA@SD   28.6